In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tiktoken
import torch
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sakshisharma221/shakespeare-text/input.txt


In [2]:
with open("/kaggle/input/datasets/sakshisharma221/shakespeare-text/input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [3]:
len(text)  #1M

1115394

In [4]:
unique_chars = sorted(list(set(text)))
print("".join(unique_chars))

stoi = {ch: i for i, ch in enumerate(unique_chars)}
itos = {i: ch for i, ch in enumerate(unique_chars)}

# stoi = {}
# itos = {}
# for i, c in enumerate(unique_chars):
#     stoi[c] = i
#     itos[i] = c


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [5]:
vocab_size = len(unique_chars) #65

In [6]:
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda s: "".join([itos[i] for i in s])

In [7]:
data = torch.tensor(encode(text), dtype=torch.long) 

In [8]:
data.shape

torch.Size([1115394])

In [9]:
data.dtype

torch.int64

In [10]:
data[:1000]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57,  1, 47, 57,  1, 41, 

In [11]:
print(decode(data[:1000].tolist()))

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [12]:
#splitting the data into training and validation sets
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [13]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [14]:
x = train_data[:block_size + 1]
for t in range(block_size):
    context = x[:t+1]
    target = x[t+1] 
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [15]:
torch.manual_seed(1337)
batch_size = 4 # independent sequence we will process in parallel
block_size = 8 # maximum context length for predictions

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, )) #start index
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x, y

xb, yb = get_batch("train")
print(xb)
print(yb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


## Making a Bigram Language Model

In [16]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # nn.embedding(num_tokens, vector_size)
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        # lookup table: for each token, stores a vector of size vocab_size
        # these vectors are learned during training

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx)
        
        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)    
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
            
    # max new tokens - how many new tokens to generate
    # generates many tokens in a loop, one at a time, each time feeding the previous output back in
    def generate(self, idx,  max_new_tokens):

        # [54, 1, 48] 
        # ->
        # [[vector], [vector], [vector]]
        # -> last one [[vector]]
        # -> [[probs vector]]
        # -> [get max idx in that prob]
        # -> append it to idxs
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # calling forward
            logits = logits[:, -1, :] # last
            probs = F.softmax(logits, dim=1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
        
    
m = BigramLanguageModel()
logits, loss = m(xb, yb)
print(logits.shape)
print(loss) # first loss is in a ideal case expected to be 4.17  because everyone should get -log(1/65)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


In [17]:
idx = torch.zeros((1, 1), dtype=torch.long) # [[0]]
generated = m.generate(idx, max_new_tokens=1000)

print(decode(generated[0].tolist()))


SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJt-wBpm&yiltNCjeO3:Cx&vvMYW-txjuAd IRFbTpJ$zkZelxZtTlHNzdXXUiQQY:qFINTOBNLI,&oTigq z.c:Cq,SDXzetn3XVjX-YBcHAUhk&PHdhcOb
nhJ?FJU?pRiOLQeUN!BxjPLiq-GJdUV'hsnla!murI!IM?SPNPq?VgC'R
pD3cLv-bxn-tL!upg
SZ!Uvdg CtxtT?hsiW:XxKIiPlagHIsr'zKSVxza?GlDWObPmRJgrIAcmspmZ&viCKot:u3qYXA:rZgv f:3Q-oiwUzqh'Z!I'zRS3SP rVchSFUIdd q?sPJpUdhMCK$VXXevXJFMl,i
YxA:gWId,EXR,iMC,$?srV$VztRwb?KpgUWFjR$zChOLm;JrDnDph
LBj,KZxJaLPgBAkyzEzSiiQb
jkSVyb$vvyQFuAUAKuzdZAJktRqUiAcPBa;AgJ;.$l3Pu!.IErMfN!PmuQbvx
xMkttN:PmJh'wNC
AUI?wNCphq-.IsCwbjxca;P-KA:r'a;pJ&q-UgOEX.cAO-p,lQ?nEsrlvmUgbEQLQh,j;iPlgZR:CJpxIBju f&!BBEHSPmnq,P -d
pjuWDPLFa!ByCSjJuERtKpph.ZP  CUEsiy'FjF$$-rJUQ?uApxlxlYe
yASBoipGLwfXelgY!a fyFPJX!JDWCoAXRJJFJOlxlvpR?OXYddZAXzkIBtp3d,vAcPlgX'pM fNMLphx&flaAcL!3F.?sBiRwLTqHzot.ttRF$Fv'bL:&x&ayFVqVAHqHxv3QzteqbcUJnERZYGwzLd,rgf&yCnERErI.IVZ WddEJAX CO'Eu!I!Lg:i-$ mIc.
xJjdLEJXVTb?Eqf IgCJcUGNSBZ3dbsEXgCPmr'XxxDXXxE

In [18]:
# training 

optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
for epoch in range(10000):
    optimizer.zero_grad() # clearing old gradients
    xb, yb = get_batch('train')  # fresh batch every step
    logits, loss = m(xb, yb) # forward pass
    loss.backward() # backward pass
    optimizer.step() # updating weigths
    
    if epoch % 100 == 0:
        print(loss)

tensor(4.5821, grad_fn=<NllLossBackward0>)
tensor(4.7168, grad_fn=<NllLossBackward0>)
tensor(4.3927, grad_fn=<NllLossBackward0>)
tensor(4.4314, grad_fn=<NllLossBackward0>)
tensor(4.6877, grad_fn=<NllLossBackward0>)
tensor(4.2215, grad_fn=<NllLossBackward0>)
tensor(4.3188, grad_fn=<NllLossBackward0>)
tensor(4.1628, grad_fn=<NllLossBackward0>)
tensor(4.0733, grad_fn=<NllLossBackward0>)
tensor(4.4119, grad_fn=<NllLossBackward0>)
tensor(4.2164, grad_fn=<NllLossBackward0>)
tensor(4.2841, grad_fn=<NllLossBackward0>)
tensor(4.0965, grad_fn=<NllLossBackward0>)
tensor(3.9716, grad_fn=<NllLossBackward0>)
tensor(3.9510, grad_fn=<NllLossBackward0>)
tensor(4.1028, grad_fn=<NllLossBackward0>)
tensor(3.5143, grad_fn=<NllLossBackward0>)
tensor(3.6007, grad_fn=<NllLossBackward0>)
tensor(3.1687, grad_fn=<NllLossBackward0>)
tensor(3.4646, grad_fn=<NllLossBackward0>)
tensor(3.5826, grad_fn=<NllLossBackward0>)
tensor(3.2645, grad_fn=<NllLossBackward0>)
tensor(3.1773, grad_fn=<NllLossBackward0>)
tensor(3.38

In [19]:
idx = torch.zeros((1, 1), dtype=torch.long) # [[0]]
generated = m.generate(idx, max_new_tokens=1000)

print(decode(generated[0].tolist()))


e winatllorkislmye
AGa dheng baninem e tat f:
ALjdnt tha mitrks ban?reth bre seon:P
Wrchumed th pul Fbrind moind se is he, tye ay sacllullo kSor?O, at qSul.
AD3xThie velarkearthe I y
NARIIfrey me fr'dJzP, soth pth HA blig:
Astencoulf ad ded,
ARl ea ad hean, tagr; mad LQEno ge at ver;Awirwn;

EJac,
Wker as:

L&M, wigit d.
PSVO:


DOUEHZ-garesammy s nd t s-VINYm'Th n:
Bothsw, fosatholl.
Fur owind y sise biuppbe w ls he thout t; is RD ig w an
A:
tl ane kaneeve,
Artcke mnas h weneyyVe.
CHowes f?ulll a't GHAchowhe lld.

wid pas
INGoingw, wilvewounknor ir l
arulourithertorer igCHavUM'sly,
YC
CD:
Wht Bouredik YQ?
FomernoBOL!LUEnderid w'd I tly ad han, m inck$Ginm;A:
INThet tis has me:
GlO: tere, wrery, shasstigsous s verne nre souk, be to e tor othot'B'tit keal tr k,
Thin
be.
AUAnnelads:
RYR:

ETous tht, illll d'd wonte,
M.

Fon, sq'sifast d:
To brithinguau ns, a; t;q-
OK-amy; vel wXIOWhthevewhallolvvep INDJUDWh wa! Fld st, than, w s; ve y oowa VO, s.
this sthuid la, the,Sme wswar frrt s,'do